# Exploratory Data Analysis

Understanding the dataset to explore how the data is present in the database and if there is no need of creating some aggregated tables that can help with :
* Vendor selection for profitability 
* Product Pricing Optimization


In [1]:
import pandas as pd
import sqlite3

In [2]:
#  Craeting Database connection
conn=sqlite3.connect('invertory.db')

In [3]:
# checking tables present in the database
tables=pd.read_sql_query("select name from sqlite_master where type='table'",conn)
tables

,name


In [4]:
for table in tables["name"]:
    print("-" * 50, f"{table}", "-" * 50)
    count = pd.read_sql(f"SELECT COUNT(*) AS count FROM {table}", conn)[
        "count"
    ].values[0]
    print("Count of records:", count)
    display(pd.read_sql(f"select * from {table} limit 5",conn))

In [5]:
purchases=pd.read_sql("select * from purchases where VendorNumber= 4466",conn)
purchases

DatabaseError: Execution failed on sql 'select * from purchases where VendorNumber= 4466': no such table: purchases

In [6]:
purchase_prices=pd.read_sql("select * from purchase_prices where VendorNumber= 4466",conn)
purchase_prices

DatabaseError: Execution failed on sql 'select * from purchase_prices where VendorNumber= 4466': no such table: purchase_prices

In [ ]:
sales=pd.read_sql("select * from sales where VendorNO= 4466",conn)
sales

,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-09,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
1,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-12,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
2,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-15,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
3,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-21,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
4,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-23,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9448,9_BLACKPOOL_5215,9,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-12-21,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
9449,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-02,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
9450,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-09,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
9451,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-23,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE


In [ ]:
vendor_invoice=pd.read_sql("select * from vendor_invoice where VendorNumber= 4466",conn)
vendor_invoice

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-19,8207,2023-12-27,2024-02-26,335,3142.33,16.97,None
2,4466,AMERICAN VINTAGE BEVERAGE,2024-01-18,8307,2024-01-03,2024-02-18,41,383.35,1.99,None
3,4466,AMERICAN VINTAGE BEVERAGE,2024-01-27,8469,2024-01-14,2024-03-11,72,673.20,3.30,None
4,4466,AMERICAN VINTAGE BEVERAGE,2024-02-04,8532,2024-01-19,2024-03-15,79,740.21,3.48,None
5,4466,AMERICAN VINTAGE BEVERAGE,2024-02-09,8604,2024-01-24,2024-03-15,347,3261.37,17.61,None
6,4466,AMERICAN VINTAGE BEVERAGE,2024-02-17,8793,2024-02-05,2024-04-02,72,675.36,3.17,None
7,4466,AMERICAN VINTAGE BEVERAGE,2024-03-01,8892,2024-02-12,2024-03-28,117,1096.05,5.15,None
8,4466,AMERICAN VINTAGE BEVERAGE,2024-03-07,8995,2024-02-19,2024-04-02,129,1209.27,5.44,None
9,4466,AMERICAN VINTAGE BEVERAGE,2024-03-12,9033,2024-02-22,2024-04-16,147,1377.87,6.61,None


In [ ]:
purchases.groupby(["Brand","PurchasePrice"]) [["Quantity","Dollars"]].sum()

,,Quantity,Dollars
Brand,PurchasePrice,,
3140,11.19,4640,51921.60
5215,9.41,4923,46325.43
5255,9.35,6215,58110.25


* The purchases table contains actual purchases data, including the data of purchase,products (brands) purchased by vendor, the amount paid (in dollars), and the quantity purchased.
* The purchases price column is derived from the purchase_prices table, which provides product-wise actual and purchase prices.The combination of vandor and brand is unique in this table.
* The vandor_invoice table aggegates data from the purchase table, summarzing quantity and dollar amounts, along with additional column for freight.
The table maintains uniqueness based on vendor and PO number.
* The sales table captures actual sales transacttion,detailing the brands purchases by vandor, the quanity sold, the selling price and the revenue earned

As the data that we need for analysis is distributed in different tables, we need to creates a summery a summary table containing:
* purchases transactions made by vandors
* sales transaction data
* freight costs for each vendor 
* actual product prices from vendors

In [ ]:
freight_summery=pd.read_sql_query("select VendorNumber,SUM(Freight) as Freight_cost from vendor_invoice GROUP BY VendorNumber",conn)
freight_summery

,VendorNumber,Freight_cost
0,2,27.08
1,54,0.48
2,60,367.52
3,105,62.39
4,200,6.19
...,...,...
121,98450,856.02
122,99166,130.09
123,172662,178.34
124,173357,202.50


In [ ]:
purchases.columns

Index(['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'VendorNumber',
       'VendorName', 'PONumber', 'PODate', 'ReceivingDate', 'InvoiceDate',
       'PayDate', 'PurchasePrice', 'Quantity', 'Dollars', 'Classification'],
      dtype='str')

In [ ]:
purchase_prices.columns

Index(['Brand', 'Description', 'Price', 'Size', 'Volume', 'Classification',
       'PurchasePrice', 'VendorNumber', 'VendorName'],
      dtype='str')

In [ ]:
query = """
SELECT 
    p.VendorNumber,
    p.VendorName,
    p.Brand,
    p.PurchasePrice,
    pp.Volume,
    pp.Price AS ActualPrices,
    SUM(p.Quantity) AS TotalPurchaseQuantity,
    SUM(p.Dollars) AS TotalDollarsQuantity 
FROM purchases p 
JOIN purchase_prices pp ON p.Brand = pp.Brand
where p.PurchasePrice>0
GROUP BY p.VendorNumber, p.VendorName, p.Brand
ORDER BY TotalPurchaseQuantity
"""

df_result = pd.read_sql_query(query, conn)
df_result

,VendorNumber,VendorName,Brand,PurchasePrice,Volume,ActualPrices,TotalPurchaseQuantity,TotalDollarsQuantity
0,54,AAPER ALCOHOL & CHEMICAL CO,990,105.07,3750,134.49,1,105.07
1,480,BACARDI USA INC,4357,7.46,750,9.99,1,7.46
2,480,BACARDI USA INC,5181,23.99,750,29.99,1,23.99
3,480,BACARDI USA INC,6916,11.10,200,14.99,1,11.10
4,516,BANFI PRODUCTS CORP,17172,4.10,750,5.99,1,4.10
...,...,...,...,...,...,...,...,...
10687,10000,MAJESTIC FINE WINES,6570,6.77,750,10.99,215668,1460072.36
10688,3960,DIAGEO NORTH AMERICA INC,3606,0.74,50,0.99,226085,167302.90
10689,8004,SAZERAC CO INC,5111,0.72,50,0.99,244654,176150.88
10690,3960,DIAGEO NORTH AMERICA INC,1892,0.72,50,0.99,254824,183473.28


In [ ]:
sales.columns

Index(['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'SalesQuantity',
       'SalesDollars', 'SalesPrice', 'SalesDate', 'Volume', 'Classification',
       'ExciseTax', 'VendorNo', 'VendorName'],
      dtype='str')

In [ ]:
query1 = """
SELECT 
    VendorNO,
    Brand,
    SUM(SalesDollars) as TotalSalesDollars,
    SUM(SalesPrice) as TotalSalesPrice,
    SUM(SalesQuantity) as TotalSalesQuantity,
    SUM(ExciseTax) as TotalExciseTax
    FROM sales
    GROUP BY VendorNO,Brand
    ORDER BY TotalSalesDollars > 0
""" 
pd.read_sql_query(query1,conn)

,VendorNo,Brand,TotalSalesDollars,TotalSalesPrice,TotalSalesQuantity,TotalExciseTax
0,2,90085,665.82,295.92,18,2.00
1,2,90609,599.76,449.82,24,0.52
2,60,771,704.53,494.67,47,37.01
3,60,3979,66871.69,41682.51,3931,7224.06
4,105,2529,359.88,59.98,12,9.44
...,...,...,...,...,...,...
11267,173357,2804,6298.60,3194.29,140,110.33
11268,173357,3666,8996.40,4873.05,360,141.19
11269,173357,3848,185.94,92.97,6,4.71
11270,173357,3909,24540.18,14469.21,982,773.87


In [ ]:
Summerize_query= """
WITH FreightSummary AS (
    SELECT 
        VendorNumber,
        SUM(Freight) AS Freight_cost 
    FROM vendor_invoice 
    GROUP BY VendorNumber
),
PurchaseSummary AS (
    SELECT 
        p.VendorNumber,
        p.VendorName,
        p.Brand,
        p.Description,
        p.PurchasePrice,
        pp.Volume,
        pp.Price AS ActualPrices,
        SUM(p.Quantity) AS TotalPurchaseQuantity,
        SUM(p.Dollars) AS TotalDollarsQuantity 
    FROM purchases p 
    JOIN purchase_prices pp ON p.Brand = pp.Brand
    WHERE p.PurchasePrice > 0
    GROUP BY p.VendorNumber, p.VendorName, p.Brand, p.Description, p.PurchasePrice, pp.Volume, pp.Price
),
SalesSummary AS (
    SELECT 
        VendorNO,
        Brand,
        SUM(SalesDollars) AS TotalSalesDollars,
        SUM(SalesPrice) AS TotalSalesPrice,
        SUM(SalesQuantity) AS TotalSalesQuantity,
        SUM(ExciseTax) AS TotalExciseTax
    FROM sales
    GROUP BY VendorNO, Brand
)
SELECT
    ps.VendorNumber,
    ps.VendorName,
    ps.Brand,
    ps.Description,
    ps.PurchasePrice,
    ps.Volume,
    ps.ActualPrices,
    ps.TotalPurchaseQuantity,
    ps.TotalDollarsQuantity,
    ss.TotalSalesDollars,
    ss.TotalSalesPrice,
    ss.TotalSalesQuantity,
    ss.TotalExciseTax,
    fs.Freight_cost
FROM PurchaseSummary ps
LEFT JOIN SalesSummary ss
    ON ps.VendorNumber = ss.VendorNO
    AND ps.Brand = ss.Brand
LEFT JOIN FreightSummary fs
    ON ps.VendorNumber = fs.VendorNumber
ORDER BY ps.TotalDollarsQuantity DESC
"""

vendor_sales_summary = pd.read_sql_query(Summerize_query, conn)

In [ ]:
vendor_sales_summary

,VendorNumber,VendorName,Brand,Description,PurchasePrice,Volume,ActualPrices,TotalPurchaseQuantity,TotalDollarsQuantity,TotalSalesDollars,TotalSalesPrice,TotalSalesQuantity,TotalExciseTax,Freight_cost
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,1750,36.99,145080,3811251.60,5101919.51,672819.31,142049.0,260999.20,68601.68
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,1750,28.99,164038,3804041.22,4819073.49,561512.37,160247.0,294438.66,144929.24
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,1750,24.99,187407,3418303.68,4538120.60,461140.15,187140.0,343854.07,123780.22
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,1750,22.99,201682,3261197.94,4475972.88,420050.01,200412.0,368242.80,257032.07
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,1750,29.99,138109,3023206.01,4223107.62,545778.28,135838.0,249587.83,257032.07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,9815,WINE GROUP INC,8527,Concannon Glen Ellen Wh Zin,1.32,750,4.99,2,2.64,15.95,10.96,5.0,0.55,27100.41
10688,8004,SAZERAC CO INC,5683,Dr McGillicuddy's Apple Pie,0.39,50,0.49,6,2.34,65.66,1.47,134.0,7.04,50293.62
10689,3924,HEAVEN HILL DISTILLERIES,9123,Deep Eddy Vodka,0.74,50,0.99,2,1.48,1.98,0.99,2.0,0.10,14069.87
10690,3960,DIAGEO NORTH AMERICA INC,6127,The Club Strawbry Margarita,1.47,200,1.99,1,1.47,143.28,77.61,72.0,15.12,257032.07


This query generates a vendor-wise sales and purcahse summary,which is valuable for:
# Performance Optimization:
* The query involves heavy joins and aggregation on large datasets like sales and purchases.
* Storing the pre-aggregated results avoids repeated expensive computations.
* Helps in analyzing sales, purchase and pricing for different vendors and brands.
* Instead of running expensive queries each time, dashboards can fetch data from vendor_sales_summary. 

In [ ]:
vendor_sales_summary.dtypes

VendorNumber               int64
VendorName                   str
Brand                      int64
Description                  str
PurchasePrice            float64
Volume                       str
ActualPrices             float64
TotalPurchaseQuantity      int64
TotalDollarsQuantity     float64
TotalSalesDollars        float64
TotalSalesPrice          float64
TotalSalesQuantity       float64
TotalExciseTax           float64
Freight_cost             float64
dtype: object

In [ ]:
vendor_sales_summary

,VendorNumber,VendorName,Brand,Description,PurchasePrice,Volume,ActualPrices,TotalPurchaseQuantity,TotalDollarsQuantity,TotalSalesDollars,TotalSalesPrice,TotalSalesQuantity,TotalExciseTax,Freight_cost,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchseRatio
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,1750.0,36.99,145080,3811251.60,5101919.51,672819.31,142049.0,260999.20,68601.68,1290667.91,-381125160.0,0.979108,1.338647
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,1750.0,28.99,164038,3804041.22,4819073.49,561512.37,160247.0,294438.66,144929.24,1015032.27,-380404122.0,0.976890,1.266830
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,1750.0,24.99,187407,3418303.68,4538120.60,461140.15,187140.0,343854.07,123780.22,1119816.92,-341830368.0,0.998575,1.327594
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,1750.0,22.99,201682,3261197.94,4475972.88,420050.01,200412.0,368242.80,257032.07,1214774.94,-326119794.0,0.993703,1.372493
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,1750.0,29.99,138109,3023206.01,4223107.62,545778.28,135838.0,249587.83,257032.07,1199901.61,-302320601.0,0.983556,1.396897
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,9815,WINE GROUP INC,8527,Concannon Glen Ellen Wh Zin,1.32,750.0,4.99,2,2.64,15.95,10.96,5.0,0.55,27100.41,13.31,-264.0,2.500000,6.041667
10688,8004,SAZERAC CO INC,5683,Dr McGillicuddy's Apple Pie,0.39,50.0,0.49,6,2.34,65.66,1.47,134.0,7.04,50293.62,63.32,-234.0,22.333333,28.059829
10689,3924,HEAVEN HILL DISTILLERIES,9123,Deep Eddy Vodka,0.74,50.0,0.99,2,1.48,1.98,0.99,2.0,0.10,14069.87,0.50,-148.0,1.000000,1.337838
10690,3960,DIAGEO NORTH AMERICA INC,6127,The Club Strawbry Margarita,1.47,200.0,1.99,1,1.47,143.28,77.61,72.0,15.12,257032.07,141.81,-147.0,72.000000,97.469388


In [ ]:
vendor_sales_summary.isnull().sum()

VendorNumber             0
VendorName               0
Brand                    0
Description              0
PurchasePrice            0
Volume                   0
ActualPrices             0
TotalPurchaseQuantity    0
TotalDollarsQuantity     0
TotalSalesDollars        0
TotalSalesPrice          0
TotalSalesQuantity       0
TotalExciseTax           0
Freight_cost             0
GrossProfit              0
dtype: int64

In [ ]:
vendor_sales_summary['VendorName'].unique()

<StringArray>
[          'BROWN-FORMAN CORP',       'MARTIGNETTI COMPANIES',
           'PERNOD RICARD USA',    'DIAGEO NORTH AMERICA INC',
             'BACARDI USA INC',     'JIM BEAM BRANDS COMPANY',
         'MAJESTIC FINE WINES',  'ULTRA BEVERAGE COMPANY LLP',
       'STOLI GROUP,(USA) LLC',        'PROXIMO SPIRITS INC.',
 ...
                    'UNCORKED',         'BRONCO WINE COMPANY',
     'MILTONS DISTRIBUTING CO',                'TRUETT HURST',
         'LAUREATE IMPORTS CO',     'FANTASY FINE WINES CORP',
 'AAPER ALCOHOL & CHEMICAL CO',      'SILVER MOUNTAIN CIDERS',
      'CAPSTONE INTERNATIONAL',          'FLAVOR ESSENCE INC']
Length: 128, dtype: str

In [ ]:
vendor_sales_summary['Description'].unique()

<StringArray>
[    'Jack Daniels No 7 Black',       'Tito's Handmade Vodka',
            'Absolut 80 Proof',      'Capt Morgan Spiced Rum',
             'Ketel One Vodka',            'Grey Goose Vodka',
       'Jameson Irish Whiskey',           'Smirnoff Traveler',
                   'Tanqueray',                    'Jim Beam',
 ...
         'Riunite Sweet White',         'Firefly Ridge Chard',
           'Debauchery Pnt Nr', 'Terruzzi & Puthod Vernaccia',
  'Santa Rita Organic Svgn Bl',    'Aresti Pnt Nr Curico Vly',
   'Smirnoff Light Strawberry',           'Crown Royal Apple',
 'Concannon Glen Ellen Wh Zin', 'The Club Strawbry Margarita']
Length: 9651, dtype: str

In [ ]:
vendor_sales_summary['Volume'] = vendor_sales_summary['Volume'].astype('float64')

In [ ]:
vendor_sales_summary .fillna(0, inplace=True)

,VendorNumber,VendorName,Brand,Description,PurchasePrice,Volume,ActualPrices,TotalPurchaseQuantity,TotalDollarsQuantity,TotalSalesDollars,TotalSalesPrice,TotalSalesQuantity,TotalExciseTax,Freight_cost,GrossProfit,ProfitMargin,StockTurnover
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,1750.0,36.99,145080,3811251.60,5101919.51,672819.31,142049.0,260999.20,68601.68,1290667.91,-381125160.0,0.979108
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,1750.0,28.99,164038,3804041.22,4819073.49,561512.37,160247.0,294438.66,144929.24,1015032.27,-380404122.0,0.976890
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,1750.0,24.99,187407,3418303.68,4538120.60,461140.15,187140.0,343854.07,123780.22,1119816.92,-341830368.0,0.998575
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,1750.0,22.99,201682,3261197.94,4475972.88,420050.01,200412.0,368242.80,257032.07,1214774.94,-326119794.0,0.993703
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,1750.0,29.99,138109,3023206.01,4223107.62,545778.28,135838.0,249587.83,257032.07,1199901.61,-302320601.0,0.983556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,9815,WINE GROUP INC,8527,Concannon Glen Ellen Wh Zin,1.32,750.0,4.99,2,2.64,15.95,10.96,5.0,0.55,27100.41,13.31,-264.0,2.500000
10688,8004,SAZERAC CO INC,5683,Dr McGillicuddy's Apple Pie,0.39,50.0,0.49,6,2.34,65.66,1.47,134.0,7.04,50293.62,63.32,-234.0,22.333333
10689,3924,HEAVEN HILL DISTILLERIES,9123,Deep Eddy Vodka,0.74,50.0,0.99,2,1.48,1.98,0.99,2.0,0.10,14069.87,0.50,-148.0,1.000000
10690,3960,DIAGEO NORTH AMERICA INC,6127,The Club Strawbry Margarita,1.47,200.0,1.99,1,1.47,143.28,77.61,72.0,15.12,257032.07,141.81,-147.0,72.000000


In [ ]:
vendor_sales_summary['VendorName'] = vendor_sales_summary['VendorName'].str.strip()

In [ ]:
vendor_sales_summary['GrossProfit'] = vendor_sales_summary['TotalSalesDollars'] - vendor_sales_summary['TotalDollarsQuantity']

In [ ]:
vendor_sales_summary['ProfitMargin'] = (vendor_sales_summary['GrossProfit'] - vendor_sales_summary['TotalSalesDollars'])*100

In [ ]:
vendor_sales_summary['StockTurnover'] = vendor_sales_summary['TotalSalesQuantity']/vendor_sales_summary['TotalPurchaseQuantity']

In [ ]:
vendor_sales_summary['SalesToPurchseRatio'] = vendor_sales_summary['TotalSalesDollars']/vendor_sales_summary['TotalDollarsQuantity']

In [ ]:
cursor = conn.cursor()

In [ ]:
vendor_sales_summary.dtypes

VendorNumber               int64
VendorName                   str
Brand                      int64
Description                  str
PurchasePrice            float64
Volume                   float64
ActualPrices             float64
TotalPurchaseQuantity      int64
TotalDollarsQuantity     float64
TotalSalesDollars        float64
TotalSalesPrice          float64
TotalSalesQuantity       float64
TotalExciseTax           float64
Freight_cost             float64
GrossProfit              float64
ProfitMargin             float64
StockTurnover            float64
SalesToPurchseRatio      float64
dtype: object

In [ ]:
vendor_sales_summary.columns

Index(['VendorNumber', 'VendorName', 'Brand', 'Description', 'PurchasePrice',
       'Volume', 'ActualPrices', 'TotalPurchaseQuantity',
       'TotalDollarsQuantity', 'TotalSalesDollars', 'TotalSalesPrice',
       'TotalSalesQuantity', 'TotalExciseTax', 'Freight_cost', 'GrossProfit',
       'ProfitMargin', 'StockTurnover', 'SalesToPurchseRatio'],
      dtype='str')

In [ ]:
cursor.execute("""CREATE TABLE vendor_sales_summary(
  VendorNumber INT,
  VendorName VARCHAR(100),
  Brand INT,
  Description VARCHAR(100),
  PurchasePrice DECIMAL(15,2),
  ActualPrices DECIMAL(15,2),
  Volume ,
  TotalPurchaseQuantity INT,
  TotalDollarsQuantity DECIMAL(15,2),
  TotalSalesQuantity INT,
  TotalSalesDollars DECIMAL(15,2),
  TotalSalesPrice DECIMAL(15,2),
  TotalExciseTax DECIMAL(15,2),
  Freight_cost DECIMAL(15,2),
  GrossProfit DECIMAL(15,2),
  ProfitMargin DECIMAL(15,2),
  StockTurnover DECIMAL(15,2),
  SalesToPurchseRatio DECIMAL(15,2),
  PRIMARY KEY (VendorNumber , Brand)
);
""")

In [ ]:
pd.read_sql_query("SELECT * FROM vendor_sales_summary",conn)

,VendorNumber,VendorName,Brand,Description,PurchasePrice,Volume,ActualPrices,TotalPurchaseQuantity,TotalDollarsQuantity,TotalSalesDollars,TotalSalesPrice,TotalSalesQuantity,TotalExciseTax,Freight_cost,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchseRatio
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,1750.0,36.99,145080,3811251.60,5101919.51,672819.31,142049.0,260999.20,68601.68,1290667.91,-381125160.0,0.979108,1.338647
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,1750.0,28.99,164038,3804041.22,4819073.49,561512.37,160247.0,294438.66,144929.24,1015032.27,-380404122.0,0.976890,1.266830
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,1750.0,24.99,187407,3418303.68,4538120.60,461140.15,187140.0,343854.07,123780.22,1119816.92,-341830368.0,0.998575,1.327594
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,1750.0,22.99,201682,3261197.94,4475972.88,420050.01,200412.0,368242.80,257032.07,1214774.94,-326119794.0,0.993703,1.372493
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,1750.0,29.99,138109,3023206.01,4223107.62,545778.28,135838.0,249587.83,257032.07,1199901.61,-302320601.0,0.983556,1.396897
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,9815,WINE GROUP INC,8527,Concannon Glen Ellen Wh Zin,1.32,750.0,4.99,2,2.64,15.95,10.96,5.0,0.55,27100.41,13.31,-264.0,2.500000,6.041667
10688,8004,SAZERAC CO INC,5683,Dr McGillicuddy's Apple Pie,0.39,50.0,0.49,6,2.34,65.66,1.47,134.0,7.04,50293.62,63.32,-234.0,22.333333,28.059829
10689,3924,HEAVEN HILL DISTILLERIES,9123,Deep Eddy Vodka,0.74,50.0,0.99,2,1.48,1.98,0.99,2.0,0.10,14069.87,0.50,-148.0,1.000000,1.337838
10690,3960,DIAGEO NORTH AMERICA INC,6127,The Club Strawbry Margarita,1.47,200.0,1.99,1,1.47,143.28,77.61,72.0,15.12,257032.07,141.81,-147.0,72.000000,97.469388


In [ ]:
vendor_sales_summary.to_sql('vendor_sales_summary',conn,if_exists='replace',index=False)

10692